# Aethera #35: FarukSTT on 20 Derja farming questions

Two configurations on the same recordings: **A** automatic language detection, **B** language forced to Arabic.
Runtime -> Change runtime type -> **T4 GPU**. Upload a .zip of the recordings and the question list as CSV (columns `id`, `question_derja`, `audio_file`).
WER is computed against the question text after light normalisation: a rough guide, not a semantic score.

In [ ]:
!pip -q install transformers accelerate librosa

In [ ]:
from google.colab import files
up = files.upload()   # choose your audio .zip and the sheet .csv (any file names)
print(list(up))

In [ ]:
import zipfile, os, glob, pandas as pd
zip_name = [f for f in up if f.lower().endswith(".zip")][0]
csv_name = [f for f in up if f.lower().endswith(".csv")][0]
os.makedirs("audio", exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("audio")
try:
    df = pd.read_csv(csv_name, encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(csv_name, encoding="cp1256")   # CSV saved by Arabic Windows Excel
print(zip_name, "|", csv_name, "|", df.shape)
print("audio files:", sorted(os.path.basename(p) for p in glob.glob("audio/**/*.*", recursive=True)))
df[["id", "question_derja", "audio_file"]].head()

In [ ]:
import torch, time
from transformers import pipeline

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
LOADED = None
for repo in ["medyas/FarukSTT", "FARUKxAUTO/FarukSTT"]:   # first one that loads
    try:
        pipe = pipeline("automatic-speech-recognition", model=repo,
                        torch_dtype=torch.float16, device=0)
        LOADED = repo
        break
    except Exception as e:
        print("failed:", repo, "->", str(e)[:200])
print("LOADED:", LOADED)   # write this repo ID in the report

In [ ]:
import re, unicodedata

_VARIANTS = {"اليوم": "ليوم", "هاذي": "هذي", "هاذا": "هذا", "برشه": "برشا", "برشة": "برشا"}
_PUNCT = re.compile(r"[^\w\s]", re.UNICODE)

def normalize(text):
    """Light normalisation so spelling variants of Derja are not counted as errors."""
    t = unicodedata.normalize("NFKD", str(text or "")).lower()
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")   # accents + Arabic diacritics
    t = t.replace("ـ", "")                                    # tatweel
    t = (t.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
           .replace("ى", "ي").replace("ة", "ه").replace("ؤ", "و").replace("ئ", "ي"))
    t = _PUNCT.sub(" ", t)
    words = [_VARIANTS.get(w, w) for w in t.split()]
    return " ".join(words)

def wer(ref, hyp):
    r, h = normalize(ref).split(), normalize(hyp).split()
    if not r:
        return 0.0 if not h else 1.0
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = d[j]
            d[j] = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
            prev = cur
    return d[len(h)] / len(r)

## Experiment A: automatic language detection

In [ ]:
import glob, subprocess, librosa

USE_LANGUAGE_HINT = False   # set True only if the output comes out in the wrong language/script
gk = {"language": "arabic", "task": "transcribe"} if USE_LANGUAGE_HINT else None

def find_audio(row):
    """Match by the id (q01) whatever the extension (.wav/.m4a/.mp3), or by the exact audio_file name."""
    for key in (os.path.splitext(str(row["audio_file"]))[0], str(row["id"])):
        hits = [h for h in glob.glob(f"audio/**/{key}.*", recursive=True) if not os.path.basename(h).startswith(".")]
        if hits:
            return hits[0]
    return None

def load_16k(path):
    try:
        return librosa.load(path, sr=16000)[0]
    except Exception:   # e.g. .m4a without a decoder: convert with ffmpeg (installed on Colab)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-ar", "16000", "-ac", "1", "tmp16k.wav"], check=True)
        return librosa.load("tmp16k.wav", sr=16000)[0]

heard, secs = [], []
for _, row in df.iterrows():
    path = find_audio(row)
    if path is None:
        heard.append("[FILE NOT FOUND]"); secs.append(None); print(row["id"], "| FILE NOT FOUND"); continue
    audio = load_16k(path)
    t0 = time.time()
    out = pipe(audio, generate_kwargs=gk) if gk else pipe(audio)
    secs.append(round(time.time() - t0, 1))
    heard.append(out["text"].strip())
    print(row["id"], "|", row["question_derja"], "->", heard[-1])

df["model_heard"] = heard
df["seconds"] = secs

In [ ]:
# Calculate WER for each question
df["wer"] = [
    wer(ref, hyp)
    for ref, hyp in zip(df["question_derja"], df["model_heard"])
]

# Calculate average WER
average_wer = df["wer"].mean()

print("Average WER:", round(average_wer * 100, 2), "%")
print("Average Accuracy (1 - WER):", round((1 - average_wer) * 100, 2), "%")

# Display individual results
print(df[["id", "question_derja", "model_heard", "wer"]].to_string(index=False))

In [ ]:
import re
def contains_hebrew(text):
    return bool(re.search(r"[\u0590-\u05FF]", str(text)))

df["hebrew_auto"] = df["model_heard"].apply(contains_hebrew)
print("Experiment A, predictions with Hebrew letters:", df["hebrew_auto"].sum(), "/", len(df))
print(df.loc[df["hebrew_auto"], ["id", "model_heard"]].to_string(index=False))

## Experiment B: Arabic language hint

In [ ]:
# Experiment B: Force Arabic as the transcription language

heard_ar = []
secs_ar = []

for _, row in df.iterrows():
    path = find_audio(row)

    if path is None:
        heard_ar.append("[FILE NOT FOUND]")
        secs_ar.append(None)
        continue

    audio = load_16k(path)

    t0 = time.time()

    out = pipe(
        audio,
        generate_kwargs={
            "language": "arabic",
            "task": "transcribe"
        }
    )

    secs_ar.append(round(time.time() - t0, 1))
    heard_ar.append(out["text"].strip())

    print(row["id"], "->", heard_ar[-1])

print("\nCompleted:", len(heard_ar), "transcriptions")

In [ ]:
# Save Experiment B predictions separately
df["model_heard_ar"] = heard_ar
df["seconds_ar"] = secs_ar

# Calculate WER for Experiment B
df["wer_ar"] = [
    wer(ref, hyp)
    for ref, hyp in zip(df["question_derja"], df["model_heard_ar"])
]

# Compare both experiments
print("EXPERIMENT A — Automatic language detection")
print("Mean WER:", round(df["wer"].mean() * 100, 2), "%")

print("\nEXPERIMENT B — Arabic language hint")
print("Mean WER:", round(df["wer_ar"].mean() * 100, 2), "%")

print("\nComparison:")
print("A:", round(df["wer"].mean() * 100, 2), "%")
print("B:", round(df["wer_ar"].mean() * 100, 2), "%")

df["hebrew_ar"] = df["model_heard_ar"].apply(contains_hebrew)
print("\nHebrew letters, A:", df["hebrew_auto"].sum(), "/", len(df), "| B:", df["hebrew_ar"].sum(), "/", len(df))
print("Mean seconds, A:", round(df["seconds"].mean(), 2), "| B:", round(df["seconds_ar"].mean(), 2))
print("\nDetailed results:")
print(df[["id", "wer", "wer_ar"]].to_string(index=False))

## Save results

In [ ]:
df.to_csv("FarukSTT_experiments_A_B.csv", index=False, encoding="utf-8-sig")
print("Model:", LOADED)   # write this repo ID in the README
from google.colab import files
files.download("FarukSTT_experiments_A_B.csv")